# Evaluation Notebook

In [1]:
import torch
import json
import re
import gc
from collections import defaultdict
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from llm_wrapper import DGQModelWrapper

LETTERS = "ABCDEFGHIJ"

In [2]:
# Model to test
model_name = "google/gemma-3-4b-it"

# Outlier Map
outlier_map_path = "outlier_map_google_gemma-3-4b-it-0.01.json"
with open(outlier_map_path, "r") as f:
    outlier_map = json.load(f)

device = "cuda"

In [ ]:
def load_model(model_name: str, outlier_map, device: str, weight_bits: int, act_bits: int):
    """Load either the base model or the DGQ-quantised model."""
    tokeniser = AutoTokenizer.from_pretrained(model_name)
    if tokeniser.pad_token_id is None:
        tokeniser.pad_token = tokeniser.eos_token
    tokeniser.padding_side = "left"  # Required for batched generation

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
    )

    if outlier_map:
        print("Applying DGQ quantisation...")
        model = DGQModelWrapper(
            base_model,
            outlier_map=outlier_map,
            weight_bits=weight_bits,
            act_bits=act_bits,
            compile_model=True
        )
    else:
        print("Running base model (no quantisation).")
        model = base_model

    model.to(device)
    model.eval()
    return model, tokeniser


def format_prompt(item):
    """Format a single MMLU-Pro item into a zero-shot prompt."""
    question = item["question"]
    options = item["options"]

    prompt = f"Question: {question}\nOptions:\n"
    for idx, option in enumerate(options):
        prompt += f"{LETTERS[idx]}. {option}\n"
    prompt += "Answer:"
    return prompt


def extract_answer(generated_text: str) -> str:
    """
    Extract the predicted answer letter from the model's generation.
    Looks for the first occurrence of a single capital letter A-J.
    """
    # Try to find a standalone letter first (e.g. "A" or "A." or "(A)")
    match = re.search(r'\b([A-J])\b', generated_text.strip())
    if match:
        return match.group(1)

    # Fallback: first capital letter A-J anywhere
    for char in generated_text.strip():
        if char in LETTERS:
            return char

    return ""


@torch.no_grad()
def evaluate(model, tokeniser, dataset, device, max_samples=None, batch_size=8):
    """Run evaluation and return per-category and overall accuracy."""
    category_correct = defaultdict(int)
    category_total = defaultdict(int)

    total_correct = 0
    total = 0

    samples = dataset
    if max_samples is not None:
        samples = dataset.select(range(min(max_samples, len(dataset))))

    for i in tqdm(range(0, len(samples), batch_size), desc="Evaluating"):
        batch = samples[i:i+batch_size]
        
        prompts = []
        expected_letters = []
        categories = batch["category"]
        
        # Sliced HF datasets return a dict of lists
        for j in range(len(batch["question"])):
            item = {k: batch[k][j] for k in batch.keys()}
            prompts.append(format_prompt(item))
            expected_letters.append(LETTERS[item["answer_index"]])

        # Tokenise and generate
        inputs = tokeniser(prompts, return_tensors="pt", truncation=True, padding=True, max_length=4096).to(device)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokeniser.pad_token_id,
        )

        # Decode only the newly generated tokens
        input_length = inputs["input_ids"].shape[-1]
        new_tokens_batch = output_ids[:, input_length:]
        generated_texts = tokeniser.batch_decode(new_tokens_batch, skip_special_tokens=True)

        for generated, correct_letter, category in zip(generated_texts, expected_letters, categories):
            predicted = extract_answer(generated)
            is_correct = predicted == correct_letter

            category_total[category] += 1
            total += 1
            if is_correct:
                category_correct[category] += 1
                total_correct += 1

    return total_correct, total, category_correct, category_total

def flush_memory():
    """Force garbage collection and empty CUDA cache."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

results = {}

# Base Model
# Load model
model, tokeniser = load_model(model_name, None, device, None, None)

# Load dataset
print(f"Loading MMLU-Pro (test split)...")
dataset = load_dataset("TIGER-Lab/MMLU-Pro")["test"]

# Evaluate
total_correct, total, cat_correct, cat_total = evaluate(
    model, tokeniser, dataset, device, max_samples=None, batch_size=8
)

# Save result
results["16_16"] = {total_correct, total}

# Free up VRAM
del model
del tokeniser
flush_memory()

for weight_bits, act_bits in [(8,8),(8,6),(4,8),(4,6)]:
    # Load model
    model, tokeniser = load_model(model_name, outlier_map, device, weight_bits, act_bits)

    # Evaluate
    total_correct, total, cat_correct, cat_total = evaluate(
        model, tokeniser, dataset, device, max_samples=None, batch_size=8
    )

    # Save result
    results[f"{weight_bits}_{act_bits}"] = {total_correct, total}

    # Free up VRAM
    del model
    del tokeniser
    flush_memory()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Running base model (no quantisation).
Loading MMLU-Pro (test split)...


Evaluating: 100%|██████████| 1504/1504 [08:26<00:00,  2.97it/s]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Applying DGQ quantisation...
Wrapping model with DGQ Linear layers...
Wrapping complete.
Compiling model with torch.compile()...
Compilation complete.


Evaluating:  25%|██▍       | 31/125 [00:13<00:39,  2.35it/s]


KeyboardInterrupt: 

In [ ]:
print(results)

{'16_16': {32, 200}, '8_8': {200, 26}, '8_6': {24, 200}, '4_8': {200, 23}, '4_6': {200, 18}}
